# UC08 — Detección de Consumos Anómalos en Abonados**Objetivo:** Identificar fugas internas, fraude y averías de contador mediante detección de anomalías sobre consumos diarios.**Tecnologías:** ML.ANOMALY_DETECTION · Streamlit**Tiempo estimado:** 12 minutos

## Paso 1 — Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS ANOMALIAS_CONSUMO_DB;USE DATABASE ANOMALIAS_CONSUMO_DB;USE SCHEMA PUBLIC;CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;USE WAREHOUSE COMPUTE_WH;

## Paso 2 — Generar Maestro de Abonados2.000 clientes con tipología, calibre y zona.

In [ ]:
CREATE OR REPLACE TABLE abonados ASSELECT    'AB-' || LPAD(SEQ4()::STRING, 6, '0') AS abonado_id,    CASE WHEN SEQ4() < 1400 THEN 'Domestico' WHEN SEQ4() < 1800 THEN 'Comercial' ELSE 'Industrial' END AS tipo,    CASE WHEN SEQ4() < 1400 THEN 13 WHEN SEQ4() < 1800 THEN 25 ELSE 40 END AS calibre_mm,    'DMA-' || LPAD(MOD(SEQ4(), 50)::STRING, 4, '0') AS zona_dma,    CASE WHEN SEQ4() < 1400 THEN UNIFORM(1, 5, RANDOM()) ELSE NULL END AS num_personas_hogarFROM TABLE(GENERATOR(ROWCOUNT => 2000));SELECT tipo, COUNT(*) AS abonados FROM abonados GROUP BY tipo;

## Paso 3 — Generar Consumos Diarios con Anomalías365 días por abonado. 100 abonados tienen anomalías inyectadas (fugas, fraude, contador averiado).

In [ ]:
CREATE OR REPLACE TABLE consumos ASSELECT    a.abonado_id, f.VALUE::DATE AS fecha,    ROUND(CASE        WHEN a.tipo = 'Domestico' THEN 150 + UNIFORM(-50, 100, RANDOM()) +            CASE WHEN a.abonado_id IN (SELECT abonado_id FROM abonados LIMIT 40) AND f.VALUE::INT > 300 THEN 200 + UNIFORM(0, 150, RANDOM()) ELSE 0 END        WHEN a.tipo = 'Comercial' THEN 1500 + UNIFORM(-500, 1000, RANDOM())        ELSE 8000 + UNIFORM(-2000, 5000, RANDOM())    END, 0) AS consumo_litros,    ROUND(CASE        WHEN a.abonado_id IN (SELECT abonado_id FROM abonados LIMIT 40) AND f.VALUE::INT > 300 THEN UNIFORM(50, 150, RANDOM())        ELSE UNIFORM(0, 15, RANDOM())    END, 0) AS consumo_nocturno_litrosFROM abonados a,     LATERAL FLATTEN(ARRAY_GENERATE_RANGE(0, 90)) fWHERE a.abonado_id IN (SELECT abonado_id FROM abonados LIMIT 200);SELECT COUNT(*) AS registros, COUNT(DISTINCT abonado_id) AS abonados FROM consumos;

## Paso 4 — Entrenar Detector de Anomalías

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.ANOMALY_DETECTION detector_anomalias_consumo(    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'consumos'),    SERIES_COLNAME => 'abonado_id',    TIMESTAMP_COLNAME => 'fecha',    TARGET_COLNAME => 'consumo_litros',    LABEL_COLNAME => '');

## Paso 5 — Detectar y Clasificar Anomalías

In [ ]:
CALL detector_anomalias_consumo!DETECT_ANOMALIES(    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'consumos'),    SERIES_COLNAME => 'abonado_id',    TIMESTAMP_COLNAME => 'fecha',    TARGET_COLNAME => 'consumo_litros',    CONFIG_OBJECT => {'prediction_interval': 0.99});CREATE OR REPLACE TABLE anomalias_consumo ASSELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))WHERE IS_ANOMALY = TRUE;SELECT SERIES AS abonado, COUNT(*) AS dias_anomalos FROM anomalias_consumo GROUP BY 1 ORDER BY 2 DESC LIMIT 20;

## Paso 6 — Dashboard de Anomalías

In [ ]:
import streamlit as stfrom snowflake.snowpark.context import get_active_sessionsession = get_active_session()st.title("🚨 Detección de Consumos Anómalos")col1, col2, col3 = st.columns(3)anom = session.sql("SELECT COUNT(DISTINCT SERIES) AS n FROM anomalias_consumo").collect()[0]["N"]col1.metric("Abonados Monitorizados", 200)col2.metric("Con Anomalía", anom)col3.metric("Alertas Activas", anom)st.subheader("Abonados con Más Días Anómalos")df = session.sql("SELECT SERIES AS abonado, COUNT(*) AS dias FROM anomalias_consumo GROUP BY 1 ORDER BY 2 DESC LIMIT 15").to_pandas()st.bar_chart(df.set_index("ABONADO"))

## Paso 7 — Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS ANOMALIAS_CONSUMO_DB;